# 08 · LoRA evaluation (v1 and v2)

Evaluates both LoRA adapters from notebook 06 on the 394-query test set and compares them with the base model (notebook 07). Same protocol: top-10 retrieval, fuzzy title match ≥ 0.85, Hit@K and MRR.

| Checkpoint | Adapter folder | Corpus embeddings (notebook 03) |
|---|---|---|
| LoRA v1 | `models/lora_v1` | `artifacts/embeddings_lora_v1.npy` |
| **LoRA v2 (final)** | `models/lora_v2` | `artifacts/embeddings_lora_v2.npy` |

Both adapters are loaded onto the base model and merged for inference. The cross-encoder reranker is **not** used here so the numbers are directly comparable to the baseline.

In [ ]:
import numpy as np
import pickle
import faiss
import pandas as pd
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer
from peft import PeftModel

## Shared helpers

In [ ]:
METADATA_PATH = "../artifacts/metadata.pkl"
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

df_test = pd.read_csv("../data/eval/test_set_394.csv")
print(f"Test queries: {len(df_test)} | Types: {df_test['query_type'].value_counts().to_dict()}")

In [ ]:
def build_index(embeddings_path: str) -> faiss.Index:
    emb = np.load(embeddings_path).astype("float32")
    faiss.normalize_L2(emb)
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    print(f"  Index: {idx.ntotal:,} vectors  dim={emb.shape[1]}")
    return idx


def fuzzy_match(a: str, b: str, threshold: float = 0.85) -> tuple[bool, float]:
    ratio = SequenceMatcher(None, a.strip().lower(), b.strip().lower()).ratio()
    return ratio >= threshold, ratio


def evaluate(model: SentenceTransformer, index: faiss.Index, label: str) -> pd.DataFrame:
    """Run evaluation and return a results DataFrame."""
    rows = []
    for _, row in df_test.iterrows():
        query = row["query"]
        prefixed = f"Represent this sentence for searching relevant passages: {query}"
        vec = model.encode([prefixed]).astype("float32")
        faiss.normalize_L2(vec)

        scores, indices = index.search(vec, 10)
        results = [
            {"title": metadata[idx]["title"], "similarity": float(score)}
            for score, idx in zip(scores[0], indices[0])
        ]

        correct_rank = match_ratio = correct_score = None
        for rank, r in enumerate(results, 1):
            ok, ratio = fuzzy_match(r["title"], row["title"])
            if ok:
                correct_rank, match_ratio, correct_score = rank, ratio, r["similarity"]
                break

        rows.append({
            "title":            row["title"],
            "year":             row.get("release_year"),
            "genre":            row.get("genre"),
            "query":            query,
            "query_type":       row["query_type"],
            "top1_result":      results[0]["title"] if results else None,
            "correct_rank":     correct_rank if correct_rank is not None else "not found",
            "match_ratio":      round(match_ratio, 3) if match_ratio else None,
            "hit@1":            int(correct_rank == 1) if correct_rank else 0,
            "hit@5":            int(correct_rank is not None and correct_rank <= 5),
            "hit@10":           int(correct_rank is not None and correct_rank <= 10),
            "reciprocal_rank":  (1 / correct_rank) if correct_rank else 0.0,
            "similarity_score": correct_score or 0.0,
        })

    df_out = pd.DataFrame(rows)
    out_path = f"../results/eval_394_{label}.csv"
    df_out.to_csv(out_path, index=False)
    print(f"Saved → {out_path}")
    return df_out


def print_metrics(df: pd.DataFrame, label: str) -> None:
    print(f"\n=== {label} ===")
    print(f"  Hit@1:  {df['hit@1'].mean():.1%}")
    print(f"  Hit@5:  {df['hit@5'].mean():.1%}")
    print(f"  Hit@10: {df['hit@10'].mean():.1%}")
    print(f"  MRR:    {df['reciprocal_rank'].mean():.1%}")
    print()
    print(f"  {'Query type':>15}   Hit@1   Hit@10")
    for qt in ["oracle", "conversational", "naturalistic", "vague"]:
        sub = df[df["query_type"] == qt]
        if len(sub):
            print(f"  {qt:>15}   {sub['hit@1'].mean():.1%}   {sub['hit@10'].mean():.1%}")

---
## LoRA v1 — `mxbai-movie-lora-fast`

Saved as a **LoRA adapter** on top of the base model (no merged weights).  
We load the base model, attach the adapter, then merge weights for fast inference.

In [ ]:
LORA_V1_PATH = "../models/lora_v1"
EMB_V1_PATH  = "../artifacts/embeddings_lora_v1.npy"

print("Loading base model …")
model_v1 = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

print("Attaching LoRA adapter and merging …")
base_transformer = model_v1._first_module().auto_model
peft_model = PeftModel.from_pretrained(base_transformer, LORA_V1_PATH)
model_v1._first_module().auto_model = peft_model.merge_and_unload()

print("Building index for LoRA v1 corpus embeddings …")
index_v1 = build_index(EMB_V1_PATH)

In [ ]:
results_v1 = evaluate(model_v1, index_v1, label="lora_v1")
print_metrics(results_v1, "LoRA v1")

---
## LoRA v2 (final) — `models/lora_v2`

Same loading procedure as v1: base model + adapter, merged for inference.

In [ ]:
LORA_V2_PATH = "../models/lora_v2"
EMB_V2_PATH  = "../artifacts/embeddings_lora_v2.npy"

print("Loading base model …")
model_v2 = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

print("Attaching LoRA adapter and merging …")
base_transformer = model_v2._first_module().auto_model
peft_model = PeftModel.from_pretrained(base_transformer, LORA_V2_PATH)
model_v2._first_module().auto_model = peft_model.merge_and_unload()

print("Building index for LoRA v2 corpus embeddings …")
index_v2 = build_index(EMB_V2_PATH)


In [ ]:
results_v2 = evaluate(model_v2, index_v2, label="lora_v2")
print_metrics(results_v2, "LoRA v2 (final)")

---
## Comparison table

In [ ]:
baseline = pd.read_csv("../results/eval_394_mxbai_base.csv")  # produced by notebook 07

comparison_rows = []
for label, df in [("Baseline (mxbai)", baseline), ("LoRA v1", results_v1), ("LoRA v2 (final)", results_v2)]:
    comparison_rows.append({
        "Model":   label,
        "Hit@1":   f"{df['hit@1'].mean():.1%}",
        "Hit@5":   f"{df['hit@5'].mean():.1%}",
        "Hit@10":  f"{df['hit@10'].mean():.1%}",
        "MRR":     f"{df['reciprocal_rank'].mean():.1%}",
    })

comp_df = pd.DataFrame(comparison_rows)


display(comp_df)

In [ ]:
# Per query-type comparison (Hit@1 / Hit@10)
print("=== Per query type: Baseline vs LoRA v2 ===")
print(f"  {'Type':>15}   {'Baseline':>20}   {'LoRA v2':>20}")
for qt in ["oracle", "conversational", "naturalistic", "vague"]:
    b  = baseline[baseline["query_type"] == qt]
    v2 = results_v2[results_v2["query_type"] == qt]
    if len(b) and len(v2):
        base_str = f"{b['hit@1'].mean():.1%} / {b['hit@10'].mean():.1%}"
        v2_str   = f"{v2['hit@1'].mean():.1%} / {v2['hit@10'].mean():.1%}"
        print(f"  {qt:>15}   {base_str:>20}   {v2_str:>20}")

In [ ]:
# Delta (LoRA v2 vs Baseline)
print("\n=== Delta (LoRA v2 − Baseline) ===")
metrics = {"Hit@1": "hit@1", "Hit@5": "hit@5", "Hit@10": "hit@10", "MRR": "reciprocal_rank"}
for label, col in metrics.items():
    delta = results_v2[col].mean() - baseline[col].mean()
    sign = "+" if delta >= 0 else ""
    print(f"  {label}: {sign}{delta:.1%}")